In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC Yellow Taxi") \
    .config("spark.driver.memory",        "8g")  \
    .config("spark.executor.memory",      "8g")  \
    .config("spark.driver.maxResultSize", "8g")  \
    .getOrCreate()

In [10]:
import os
import requests

def get_data_from_to(FromMonth, FromYear, ToMonth, ToYear, FolderPath):
    if not (1 <= FromMonth <= 12 and 1 <= ToMonth <= 12):
      raise ValueError("Month must be between 1 and 12.")

    if not (2009 <= FromYear <= 2026 and 2009 <= ToYear <= 2026):
      raise ValueError("Year must be between 2009 and 2026.")

    from_date = (FromYear, FromMonth)
    to_date   = (ToYear, ToMonth)

    if from_date > to_date:
        raise ValueError(
            f"'From' date ({FromMonth}/{FromYear}) "
            f"must be before 'To' date ({ToMonth}/{ToYear}).")

    all_files = os.listdir(FolderPath)

    current_month = FromMonth
    last_month = 12

    for year in range(FromYear, ToYear + 1):

      if year == ToYear:
        last_month = ToMonth

      for month in range(current_month, last_month + 1):
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
        filename = f"{FolderPath}/yellow_tripdata_{year}-{month:02d}.parquet"

        if filename[len(FolderPath)+1:] not in all_files:
          print(f"fetching {filename}")
          download_taxi_data(filename, url, FolderPath)

      current_month = 1

def download_taxi_data(filename, url, save_path):

  response = requests.get(url, stream=True)
  response.raise_for_status()

  with open(filename, "wb") as f:
      for chunk in response.iter_content(chunk_size=8192):
          f.write(chunk)
  print(f"Downloaded: {filename}")

In [ ]:
get_data_from_to(1, 2023, 12, 2023, '/content/drive/MyDrive/PySparkprojectData')

In [ ]:
df = spark.read.parquet("/content/drive/MyDrive/PySparkprojectData")

In [ ]:
df.count()

44674399

In [ ]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [ ]:
df.show()

Py4JJavaError: An error occurred while calling o31.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.PARQUET_COLUMN_DATA_TYPE_MISMATCH] Encountered error while reading file file:///content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-10.parquet. Data type mismatches when reading Parquet column [passenger_count]. Expected Spark type double, actual Parquet type INT64. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.parquetColumnDataTypeMismatchError(QueryExecutionErrors.scala:847)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:138)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:695)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sq

In [2]:
from pyspark.sql.types import *

target_schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       DoubleType()),
    StructField("trip_distance",         DoubleType()),
    StructField("RatecodeID",            DoubleType()),
    StructField("store_and_fwd_flag",    StringType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("extra",                 DoubleType()),
    StructField("mta_tax",               DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("tolls_amount",          DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount",          DoubleType()),
    StructField("congestion_surcharge",  DoubleType()),
    StructField("airport_fee",           DoubleType()),
])

In [6]:
from functools import reduce
from pyspark.sql.functions import col, lit

dfs = []

for file_name in os.listdir('/content/drive/MyDrive/PySparkprojectData'):
  df = spark.read.parquet(os.path.join('/content/drive/MyDrive/PySparkprojectData', file_name))

  for field in target_schema:
      if field.name not in df.columns:
          df = df.withColumn(field.name, lit(None).cast(field.dataType))

  df = df.select([
      col(field.name).cast(field.dataType).alias(field.name)
      for field in target_schema
  ])

  dfs.append(df)

final_df = reduce(lambda x, y: x.unionByName(y), dfs)

In [7]:
final_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [ ]:
final_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-12-01 00:37:35|  2022-12-01 00:47:35|            1.0|          2.0|       1.0|                 N|         170|         237|           1|        8.5|  3.0|    0.5|       3.

In [ ]:
def read_files(FolderPath):
  global target_schema

  dfs = []

  for file_name in os.listdir(FolderPath):
    df = spark.read.parquet(os.path.join(FolderPath, file_name))

    for field in target_schema:
        if field.name not in df.columns:
            df = df.withColumn(field.name, lit(None).cast(field.dataType))

    df = df.select([
        col(field.name).cast(field.dataType).alias(field.name)
        for field in target_schema
    ])

    dfs.append(df)

  final_df = reduce(lambda x, y: x.unionByName(y), dfs)

  return final_df

In [15]:
from pyspark.sql import functions as F

total = final_df.count()

null_counts = final_df.select([
    F.round(
        F.sum(F.col(c).isNull().cast("int")) / total * 100, 2
    ).alias(c)
    for c in final_df.columns
])

null_counts.show(vertical=True)

-RECORD 0--------------------
 VendorID              | 0.0 
 tpep_pickup_datetime  | 0.0 
 tpep_dropoff_datetime | 0.0 
 passenger_count       | 0.0 
 trip_distance         | 0.0 
 RatecodeID            | 0.0 
 store_and_fwd_flag    | 0.0 
 PULocationID          | 0.0 
 DOLocationID          | 0.0 
 payment_type          | 0.0 
 fare_amount           | 0.0 
 extra                 | 0.0 
 mta_tax               | 0.0 
 tip_amount            | 0.0 
 tolls_amount          | 0.0 
 improvement_surcharge | 0.0 
 total_amount          | 0.0 
 congestion_surcharge  | 0.0 



In [8]:
final_df = final_df.drop("airport_fee")

final_df = final_df.dropna(how="any")

In [11]:
get_data_from_to(1, 2024, 1, 2024, '/content/drive/MyDrive/PySparkprojectData')

In [12]:
test_df = spark.read.parquet('/content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2024-01.parquet')

In [13]:
test_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [16]:
test_df = test_df.select([
    F.col(field.name).cast(field.dataType).alias(field.name)
    for field in target_schema.fields
])

In [17]:
test_df = test_df.drop("airport_fee")
test_df = test_df.dropna(how = 'any')

In [18]:
test_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [19]:
test_df.count()

2824462

# Modeling

## Modeling Fare

### LR

In [20]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline


categorical_cols = ["store_and_fwd_flag"]
indexed_cols     = [f"{c}_idx" for c in categorical_cols]

numeric_cols = ["passenger_count", "trip_distance", "PULocationID", "DOLocationID", "RatecodeID", "VendorID", 'payment_type']

indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=indexed_cols,
    handleInvalid="skip")

assembler = VectorAssembler(
    inputCols=indexed_cols + numeric_cols,
    outputCol="features")

preprocessing_pipeline = Pipeline(stages=[indexer, assembler])

preprocessing_model = preprocessing_pipeline.fit(final_df)
df_prepared = preprocessing_model.transform(final_df)

In [ ]:
import time
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="fare_amount"
    )

start_time = time.time()
lr_model = lr.fit(df_prepared)
elapsed = time.time() - start_time

print(elapsed)

352.1224641799927


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

test_prepared = preprocessing_model.transform(test_df)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

predictions = lr_model.transform(test_prepared)

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 18.7776
MAE  : 11.5916
R²   : 0.0428


In [ ]:
numeric_cols_2 = ["trip_distance", "RatecodeID"]

assembler_2 = VectorAssembler(
    inputCols = numeric_cols_2,
    outputCol = "features")

preprocessing_pipeline_2 = Pipeline(stages=[assembler_2])

preprocessing_model_2 = preprocessing_pipeline_2.fit(final_df)
df_prepared_2 = preprocessing_model_2.transform(final_df)

In [ ]:
import time
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="fare_amount"
    )

start_time = time.time()
lr_model = lr.fit(df_prepared_2)
elapsed = time.time() - start_time

print(elapsed)

126.15788793563843


In [ ]:
test_prepared_2 = preprocessing_model_2.transform(test_df)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

predictions = lr_model.transform(test_prepared_2)

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 19.0597
MAE  : 11.7999
R²   : 0.0138


### XGB

In [ ]:
from xgboost.spark import SparkXGBRegressor

xgb = SparkXGBRegressor( features_col="features", label_col="fare_amount" , device = 'cuda')

start_time = time.time()
xgb_model  = xgb.fit(df_prepared)
elapsed    = time.time() - start_time

print(elapsed)

INFO:XGBoost-PySpark:Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cuda', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


327.4529302120209


In [ ]:
predictions = xgb_model.transform(test_prepared)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 9.6394
MAE  : 2.5940
R²   : 0.7478


In [ ]:
final_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2022-12-01 00:37:35|  2022-12-01 00:47:35|            1.0|          2.0|       1.0|                 N|         170|         237|           1|        8.5|  3.0|    0.5|       3.1|         0.0|                  0.3

In [ ]:
xgb = SparkXGBRegressor( features_col="features", label_col="fare_amount", device = 'cuda')

start_time = time.time()
xgb_model  = xgb.fit(df_prepared_2)
elapsed    = time.time() - start_time

print(elapsed)

INFO:XGBoost-PySpark:Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cuda', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


230.0498824119568


In [ ]:
predictions = xgb_model.transform(test_prepared_2)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 10.1719
MAE  : 2.7614
R²   : 0.7191


In [22]:
RAW_FEATURES = [
    "trip_distance", "passenger_count", "RatecodeID",
    "PULocationID", "DOLocationID",
]
TARGET = "fare_amount"

In [38]:
final_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2022-12-01 00:37:35|  2022-12-01 00:47:35|            1.0|          2.0|       1.0|                 N|         170|         237|           1|        8.5|  3.0|    0.5|       3.1|         0.0|                  0.3

In [23]:
import math

def engineer(d):
    d = (
        d.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    )
    d = (
        d.withColumn("is_weekend", F.col("pickup_dayofweek").isin(1, 7).cast("int"))
        .withColumn(
            "is_rush_hour",
            (
                (~F.col("pickup_dayofweek").isin(1, 7))
                & (F.col("pickup_hour").between(7, 9) | F.col("pickup_hour").between(16, 19))
            ).cast("int"),
        )
        .withColumn(
            "is_airport",
            (F.col("PULocationID").isin(1, 132, 138) | F.col("DOLocationID").isin(1, 132, 138)).cast("int"),
        )
        .withColumn("log_distance", F.log1p("trip_distance"))
    )
    d = (
        d.withColumn("hour_sin", F.sin(2 * math.pi * F.col("pickup_hour") / 24))
        .withColumn("hour_cos", F.cos(2 * math.pi * F.col("pickup_hour") / 24))
        .withColumn("dow_sin", F.sin(2 * math.pi * (F.col("pickup_dayofweek") - 1) / 7))
        .withColumn("dow_cos", F.cos(2 * math.pi * (F.col("pickup_dayofweek") - 1) / 7))
    )
    return d

train_fe = engineer(final_df)
test_fe = engineer(test_df)

FE_FEATURES = RAW_FEATURES + [
    "pickup_hour", "pickup_dayofweek", "pickup_month",
    "is_weekend", "is_rush_hour", "is_airport", "log_distance",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]

In [ ]:
assembler_3 = VectorAssembler(
    inputCols=FE_FEATURES,
    outputCol="features")

preprocessing_pipeline_3 = Pipeline(stages=[assembler_3])

preprocessing_model_3 = preprocessing_pipeline_3.fit(train_fe)
df_prepared_3 = preprocessing_model_3.transform(train_fe)

In [ ]:
xgb = SparkXGBRegressor( features_col="features", label_col="fare_amount", device = 'cuda')

start_time = time.time()
xgb_model  = xgb.fit(df_prepared_3)
elapsed    = time.time() - start_time

print(elapsed)

INFO:XGBoost-PySpark:Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cuda', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


421.2303247451782


In [ ]:
test_prepared_3 = preprocessing_model_3.transform(test_fe)

predictions = xgb_model.transform(test_prepared_3)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 9.0965
MAE  : 2.4065
R²   : 0.7754


In [24]:
train_fe_2 = train_fe.filter(
    (F.col("fare_amount") >= 2.5) & (F.col("fare_amount") <= 250)
    & (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100)
    & (F.col("passenger_count") > 0)
)

train_fe_2 = train_fe_2.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

test_fe_2 = test_fe.filter(
    (F.col("fare_amount") >= 2.5) & (F.col("fare_amount") <= 250)
    & (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100)
    & (F.col("passenger_count") > 0)
)

test_fe_2 = test_fe_2.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))


In [25]:
assembler_3 = VectorAssembler(
    inputCols=FE_FEATURES,
    outputCol="features")

preprocessing_pipeline_3 = Pipeline(stages=[assembler_3])

preprocessing_model_3 = preprocessing_pipeline_3.fit(train_fe_2)
df_prepared_5 = preprocessing_model_3.transform(train_fe_2)

In [30]:
xgb = SparkXGBRegressor( features_col="features", label_col="fare_amount", device = 'cuda')

start_time = time.time()
xgb_model  = xgb.fit(df_prepared_5)
elapsed    = time.time() - start_time

print(elapsed)

INFO:XGBoost-PySpark:Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cuda', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


417.2087631225586


In [40]:
test_prepared_5 = preprocessing_model_3.transform(test_fe_2)

predictions = xgb_model.transform(test_prepared_5)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 3.1062
MAE  : 1.6122
R²   : 0.9660


In [44]:
preprocessing_model_3.write().overwrite().save("/content/drive/MyDrive/PySparkprojectData/preprocessing_pipeline")

In [45]:
xgb_model.write().overwrite().save('/content/drive/MyDrive/PySparkprojectData/xgb_model')

In [36]:
train_fe.count()

43098418

In [34]:
train_fe_2.count()

41467820

In [37]:
test_fe.count()

2824462

In [35]:
test_fe_2.count()

2723289

## modeling Tip

In [ ]:
FE_FEATURES_2 = RAW_FEATURES + [
    "pickup_hour", "pickup_dayofweek", "pickup_month",
    "is_weekend", "is_rush_hour", "is_airport", "log_distance",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
] + ['fare_amount']

assembler_4 = VectorAssembler(
    inputCols=FE_FEATURES_2,
    outputCol="features")

preprocessing_pipeline_4 = Pipeline(stages=[assembler_4])

preprocessing_model_4 = preprocessing_pipeline_4.fit(train_fe)
df_prepared_4 = preprocessing_model_4.transform(train_fe)

xgb = SparkXGBRegressor( features_col="features", label_col="tip_amount", device = 'cuda')

start_time = time.time()
xgb_model  = xgb.fit(df_prepared_4)
elapsed    = time.time() - start_time

print(elapsed)

INFO:XGBoost-PySpark:Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cuda', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


446.49032068252563


In [ ]:
test_prepared_4 = preprocessing_model_4.transform(test_fe)

predictions = xgb_model.transform(test_prepared_4)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

RMSE : 22.5556
MAE  : 15.2321
R²   : -0.3811
